## Phase 7 — AI Agent with Tools
**Model:** `databricks-meta-llama-3-3-70b-instruct` (Foundation Models API)
**Tools — Read:**
- `get_price_data` — current price + metrics from Gold
- `get_sentiment` — sentiment signal + confidence from Gold
- `compare_tickers` — side-by-side comparison of multiple tickers
- `get_top_movers` — best and worst performers today
- `search_news` — semantic RAG over news articles (Vector Search)

**Tools — Write:**
- `add_to_watchlist` — saves ticker to user watchlist (Delta)
- `save_research_note` — persists user note per ticker (Delta)
- `save_analysis_report` — logs agent-generated report (Delta)


In [ ]:
# 0. Install OpenAI SDK for Foundation Models API
%pip install openai databricks-ai-search --quiet
dbutils.library.restartPython()


In [ ]:
# 1. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from openai import OpenAI
from datetime import datetime
import json
import uuid

spark = SparkSession.builder.getOrCreate()

# Databricks Foundation Models API — OpenAI-compatible endpoint
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl", "")
TOKEN         = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
MODEL         = "databricks-meta-llama-3-3-70b-instruct"
USER_EMAIL    = "jayanthdolai07@gmail.com"

client = OpenAI(
    api_key  = TOKEN,
    base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
)

print(f"Workspace : {WORKSPACE_URL}")
print(f"Model     : {MODEL}")
print(f"User      : {USER_EMAIL}")


In [ ]:
# 2. Create agent write tables (Delta — Lakebase connection not available on Free Edition)
print("\n--- Creating agent write tables ---")

spark.sql("CREATE SCHEMA IF NOT EXISTS main.agent")

spark.sql("""
    CREATE TABLE IF NOT EXISTS main.agent.watchlists (
        id           STRING,
        user_email   STRING,
        watchlist    STRING,
        ticker       STRING,
        added_at     TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS main.agent.research_notes (
        id         STRING,
        user_email STRING,
        ticker     STRING,
        note       STRING,
        created_at TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS main.agent.analysis_reports (
        id           STRING,
        user_email   STRING,
        ticker       STRING,
        report_text  STRING,
        agent_model  STRING,
        generated_at TIMESTAMP
    ) USING DELTA
""")

print("main.agent.watchlists       ✓")
print("main.agent.research_notes   ✓")
print("main.agent.analysis_reports ✓")


In [ ]:
# 3. Tool implementations (Python functions the agent can call)

def get_price_data(ticker: str) -> dict:
    """Get current price data and metrics for a ticker."""
    try:
        rows = spark.table("main.gold.ticker_daily_summary") \
                    .filter(F.col("ticker") == ticker.upper()) \
                    .select("ticker","name","close","open","high","low",
                            "volume","daily_return_pct","price_range",
                            "is_up_day","market_cap_billions","sector") \
                    .collect()
        if not rows:
            return {"error": f"No price data found for {ticker}"}
        r = rows[0].asDict()
        return {k: (float(v) if isinstance(v, float) else v) for k, v in r.items()}
    except Exception as e:
        return {"error": str(e)}

def get_sentiment(ticker: str) -> dict:
    """Get sentiment signal and news summary for a ticker."""
    try:
        rows = spark.table("main.gold.sentiment_summary") \
                    .filter(F.col("ticker") == ticker.upper()) \
                    .collect()
        if not rows:
            return {"error": f"No sentiment data found for {ticker}"}
        r = rows[0].asDict()
        return {k: (float(v) if isinstance(v, float) else v) for k, v in r.items()}
    except Exception as e:
        return {"error": str(e)}

def compare_tickers(tickers: list) -> list:
    """Compare multiple tickers side by side."""
    try:
        ticker_list = [t.upper() for t in tickers]
        rows = spark.table("main.gold.ticker_daily_summary") \
                    .filter(F.col("ticker").isin(ticker_list)) \
                    .select("ticker","name","close","daily_return_pct",
                            "market_cap_billions","avg_sentiment_score",
                            "news_count","is_up_day") \
                    .orderBy(F.col("daily_return_pct").desc()) \
                    .collect()
        return [
            {k: (float(v) if isinstance(v, float) else v) for k, v in r.asDict().items()}
            for r in rows
        ]
    except Exception as e:
        return [{"error": str(e)}]

def get_top_movers(limit: int = 5) -> dict:
    """Get top gainers and losers for today."""
    try:
        movers = spark.table("main.gold.top_movers") \
                      .select("return_rank","ticker","name","close",
                              "daily_return_pct","mover_type","avg_sentiment_score") \
                      .collect()
        gainers = [r.asDict() for r in movers if r["mover_type"] == "GAINER"][:limit]
        losers  = [r.asDict() for r in movers if r["mover_type"] == "LOSER"][:limit]
        return {"gainers": gainers, "losers": losers}
    except Exception as e:
        return {"error": str(e)}

def search_news(query: str, num_results: int = 3) -> list:
    """Semantic search over news articles using Vector Search."""
    try:
        from databricks.ai_search.client import VectorSearchClient
        vsc = VectorSearchClient(disable_notice=True)
        idx = vsc.get_index("stock-assistant-vs", "main.silver.news_for_search_index")
        results = idx.similarity_search(
            query_text  = query,
            columns     = ["ticker","title","description","sentiment","published_utc"],
            num_results = num_results
        )
        hits = results.get("result", {}).get("data_array", [])
        cols = ["ticker","title","description","sentiment","published_utc"]
        return [dict(zip(cols, hit)) for hit in hits]
    except Exception as e:
        # Fallback: keyword search in Silver if Vector Search not ready
        rows = spark.table("main.silver.news_articles") \
                    .filter(F.lower(F.col("title")).contains(query.lower()) |
                            F.lower(F.coalesce(F.col("description"), F.lit(""))).contains(query.lower())) \
                    .select("ticker","title","description","sentiment","published_utc") \
                    .limit(num_results) \
                    .collect()
        return [r.asDict() for r in rows] if rows else [{"note": f"Search fallback used. VS error: {str(e)[:100]}"}]

def add_to_watchlist(ticker: str, watchlist_name: str = "My Watchlist") -> dict:
    """Add a ticker to user's watchlist."""
    try:
        row = spark.createDataFrame([{
            "id"          : str(uuid.uuid4()),
            "user_email"  : USER_EMAIL,
            "watchlist"   : watchlist_name,
            "ticker"      : ticker.upper(),
            "added_at"    : datetime.now().isoformat()
        }])
        row.withColumn("added_at", F.to_timestamp("added_at")) \
           .write.format("delta").mode("append") \
           .saveAsTable("main.agent.watchlists")
        return {"status": "success", "message": f"{ticker.upper()} added to '{watchlist_name}'"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def save_research_note(ticker: str, note: str) -> dict:
    """Save a research note for a ticker."""
    try:
        row = spark.createDataFrame([{
            "id"        : str(uuid.uuid4()),
            "user_email": USER_EMAIL,
            "ticker"    : ticker.upper(),
            "note"      : note,
            "created_at": datetime.now().isoformat()
        }])
        row.withColumn("created_at", F.to_timestamp("created_at")) \
           .write.format("delta").mode("append") \
           .saveAsTable("main.agent.research_notes")
        return {"status": "success", "message": f"Note saved for {ticker.upper()}"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def save_analysis_report(ticker: str, report_text: str) -> dict:
    """Save an agent-generated analysis report."""
    try:
        row = spark.createDataFrame([{
            "id"          : str(uuid.uuid4()),
            "user_email"  : USER_EMAIL,
            "ticker"      : ticker.upper(),
            "report_text" : report_text,
            "agent_model" : MODEL,
            "generated_at": datetime.now().isoformat()
        }])
        row.withColumn("generated_at", F.to_timestamp("generated_at")) \
           .write.format("delta").mode("append") \
           .saveAsTable("main.agent.analysis_reports")
        return {"status": "success", "message": f"Report saved for {ticker.upper()}"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

def get_sector_rankings() -> list:
    """Get sector rankings by total market cap, avg return and sentiment."""
    try:
        rows = spark.table("main.gold.sector_rankings") \
                    .orderBy(F.col("total_market_cap_billions").desc()) \
                    .collect()
        return [
            {k: (float(v) if isinstance(v, float) else v) for k, v in r.asDict().items()}
            for r in rows
        ]
    except Exception as e:
        return [{"error": str(e)}]


def flag_price_moves(ticker: str = None, threshold_pct: float = 2.0) -> dict:
    """Flag notable end-of-day price moves.

    Single contract, shared verbatim with app/app.py:
      - ticker given   -> check that one ticker against the threshold
      - ticker omitted -> scan every tracked ticker for moves beyond the threshold
    """
    try:
        threshold = abs(float(threshold_pct))
    except (TypeError, ValueError):
        threshold = 2.0

    def as_mover(r: dict) -> dict:
        ret = float(r.get("daily_return_pct") or 0.0)
        return {
            "ticker"          : r["ticker"],
            "name"            : r.get("name"),
            "close"           : float(r.get("close") or 0.0),
            "daily_return_pct": ret,
            "direction"       : "UP" if ret > 0 else ("DOWN" if ret < 0 else "FLAT"),
        }

    try:
        df = (spark.table("main.gold.ticker_daily_summary")
                   .select("ticker", "name", "close", "daily_return_pct",
                           "is_up_day", "snapshot_date"))

        if ticker:
            rows = df.filter(F.col("ticker") == ticker.upper()).collect()
            if not rows:
                return {"error": f"No data found for {ticker}"}
            m       = as_mover(rows[0].asDict())
            flagged = abs(m["daily_return_pct"]) >= threshold
            return {
                "threshold_pct": threshold,
                "flagged"      : flagged,
                "movers"       : [m] if flagged else [],
                "message"      : (
                    f"\u26a0\ufe0f {m['ticker']} moved {m['daily_return_pct']:+.2f}% "
                    f"\u2014 exceeds \u00b1{threshold}% threshold"
                    if flagged else
                    f"\u2705 {m['ticker']} moved {m['daily_return_pct']:+.2f}% "
                    f"\u2014 within \u00b1{threshold}% threshold"
                ),
            }

        rows = (df.filter(F.abs(F.col("daily_return_pct")) >= threshold)
                  .orderBy(F.abs(F.col("daily_return_pct")).desc())
                  .limit(10)
                  .collect())
        movers = [as_mover(r.asDict()) for r in rows]
        return {
            "threshold_pct": threshold,
            "flagged"      : bool(movers),
            "movers"       : movers,
            "message"      : (
                f"\u26a0\ufe0f {len(movers)} notable move(s) \u2265{threshold}% detected"
                if movers else
                f"\u2705 No notable price moves \u2265{threshold}% detected"
            ),
        }
    except Exception as e:
        return {"error": str(e)}

def remove_from_watchlist(ticker: str, watchlist_name: str = "My Watchlist") -> dict:
    """Remove a ticker from the user's watchlist."""
    try:
        from delta.tables import DeltaTable
        dt = DeltaTable.forName(spark, "main.agent.watchlists")
        dt.delete(
            (F.col("ticker")     == ticker.upper()) &
            (F.col("watchlist")  == watchlist_name) &
            (F.col("user_email") == USER_EMAIL)
        )
        return {"status": "success", "message": f"{ticker.upper()} removed from '{watchlist_name}'"}
    except Exception as e:
        return {"status": "error", "message": str(e)}

print("All 11 tool functions defined ✓")


In [ ]:
# 4. Tool schemas (OpenAI function-calling format)
TOOLS = [
    # ── READ TOOLS (7) ───────────────────────────────────────────────────
    {
        "type": "function",
        "function": {
            "name": "get_price_data",
            "description": "Get current price data, daily return, and key metrics for a stock ticker",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol e.g. AAPL"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_sentiment",
            "description": "Get news sentiment signal (BULLISH/NEUTRAL/BEARISH) and confidence for a ticker",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_tickers",
            "description": "Compare multiple stock tickers side by side on price and sentiment",
            "parameters": {
                "type": "object",
                "properties": {
                    "tickers": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of ticker symbols to compare"
                    }
                },
                "required": ["tickers"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_top_movers",
            "description": "Get today's top gaining and losing stocks",
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {"type": "integer", "description": "Number of results per category", "default": 5}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_news",
            "description": "Semantic search over recent news articles about stocks",
            "parameters": {
                "type": "object",
                "properties": {
                    "query"      : {"type": "string",  "description": "Search query describing what news to find"},
                    "num_results": {"type": "integer", "description": "Number of articles to return", "default": 3}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_sector_rankings",
            "description": "Get sector rankings by total market cap, avg daily return, and sentiment",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "flag_price_moves",
            "description": "Flag notable end-of-day price moves. Omit ticker to scan every tracked stock; pass a ticker to check just that one.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker"       : {"type": "string",  "description": "Optional — check a single ticker instead of scanning all"},
                    "threshold_pct": {"type": "number",  "description": "Movement threshold in percent (default 2.0)", "default": 2.0}
                }
            }
        }
    },
    # ── WRITE TOOLS (4) ──────────────────────────────────────────────────
    {
        "type": "function",
        "function": {
            "name": "add_to_watchlist",
            "description": "Add a stock ticker to the user's watchlist",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker"        : {"type": "string", "description": "Stock ticker symbol"},
                    "watchlist_name": {"type": "string", "description": "Name of the watchlist", "default": "My Watchlist"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "remove_from_watchlist",
            "description": "Remove a stock ticker from the user's watchlist",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker"        : {"type": "string", "description": "Stock ticker symbol"},
                    "watchlist_name": {"type": "string", "description": "Name of the watchlist", "default": "My Watchlist"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_research_note",
            "description": "Save a research note about a stock ticker",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "Stock ticker symbol"},
                    "note"  : {"type": "string", "description": "Research note content"}
                },
                "required": ["ticker", "note"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_analysis_report",
            "description": "Save a full analysis report for a stock ticker",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker"     : {"type": "string", "description": "Stock ticker symbol"},
                    "report_text": {"type": "string", "description": "Full report text"}
                },
                "required": ["ticker", "report_text"]
            }
        }
    },
]

TOOL_MAP = {
    # Read
    "get_price_data"      : get_price_data,
    "get_sentiment"       : get_sentiment,
    "compare_tickers"     : compare_tickers,
    "get_top_movers"      : get_top_movers,
    "search_news"         : search_news,
    "get_sector_rankings" : get_sector_rankings,
    "flag_price_moves"    : flag_price_moves,
    # Write
    "add_to_watchlist"    : add_to_watchlist,
    "remove_from_watchlist": remove_from_watchlist,
    "save_research_note"  : save_research_note,
    "save_analysis_report": save_analysis_report,
}

print(f"Registered {len(TOOLS)} tools ✓")  # 11


In [ ]:
# 5. Agent runner — agentic loop with tool execution
SYSTEM_PROMPT = """You are an AI stock market research assistant with access to end-of-day
market data, news sentiment, and portfolio management tools.

Your price data is END-OF-DAY: the most recent close from the last completed trading
day, not live intraday prices. Never describe figures as real-time, live, or current —
say "as of the latest close".

You help users:
- Analyze stock prices, returns, and sentiment
- Compare stocks and find top movers
- Search recent news articles semantically
- Manage watchlists and save research notes

Always use tools to get current data before answering. After analysis, 
offer to save a research note or add to watchlist if relevant.
Be concise and data-driven in your responses."""

def run_agent(user_query: str, verbose: bool = True) -> str:
    """
    Agentic loop:
    1. Send user query + tools to LLM
    2. If LLM calls a tool → execute it → send result back
    3. Repeat until LLM responds without tool call
    4. Return final response
    """
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_query}
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"User: {user_query}")
        print(f"{'='*60}")

    max_rounds = 5
    for round_num in range(max_rounds):
        response = client.chat.completions.create(
            model    = MODEL,
            messages = messages,
            tools    = TOOLS,
            tool_choice = "auto"
        )

        msg = response.choices[0].message

        # No tool call — final response
        if not msg.tool_calls:
            if verbose:
                print(f"\nAgent: {msg.content}")
            return msg.content

        # Execute tool calls
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ]})

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)

            if verbose:
                print(f"\n  → Tool: {fn_name}({fn_args})")

            if fn_name in TOOL_MAP:
                result = TOOL_MAP[fn_name](**fn_args)
            else:
                result = {"error": f"Unknown tool: {fn_name}"}

            if verbose:
                result_str = json.dumps(result, default=str)
                print(f"  ← Result: {result_str[:200]}{'...' if len(result_str) > 200 else ''}")

            messages.append({
                "role"       : "tool",
                "tool_call_id": tc.id,
                "content"    : json.dumps(result, default=str)
            })

    return "Max rounds reached"

print("Agent runner ready ✓")


In [ ]:
# 6. Test the agent with sample queries
print("\n=== Testing AI Agent ===\n")

# Test 1 — Read: price + sentiment
run_agent("What is Apple's current stock price and how is market sentiment?")


In [ ]:
# Test 2 — Read: comparison
run_agent("Compare MSFT and NVDA — which one has better momentum today?")


In [ ]:
# Test 3 — Read: top movers
run_agent("What are today's top gainers and losers?")


In [ ]:
# Test 4 — Read + Write: news search + save note
run_agent("Search for news about AI technology stocks and save a note about NVDA based on what you find")


In [ ]:
# Test 5 — Write: watchlist
run_agent("Add Apple and Microsoft to my Tech Watchlist")


In [ ]:
# 7. Verify write operations
print("\n=== Agent Write Verification ===")

print("\nWatchlist entries:")
spark.table("main.agent.watchlists") \
     .select("ticker", "watchlist", "user_email", "added_at") \
     .show(truncate=False)

print("\nResearch notes:")
spark.table("main.agent.research_notes") \
     .select("ticker", "note", "created_at") \
     .show(truncate=False)

print("\nAnalysis reports:")
spark.table("main.agent.analysis_reports") \
     .select("ticker", "agent_model", "generated_at") \
     .show(truncate=False)


In [ ]:
# 8. Summary
print("\n=== Phase 7 Agent Summary ===")

print(f"Model     : {MODEL}")
print(f"Tools     : {len(TOOLS)} (7 read + 4 write)")
print()

tables = ["watchlists", "research_notes", "analysis_reports"]
for t in tables:
    count = spark.table(f"main.agent.{t}").count()
    print(f"  main.agent.{t:<20} rows: {count}")

print("""
Agent capabilities:
  READ  → get_price_data, get_sentiment, compare_tickers,
           get_top_movers, search_news (RAG), get_sector_rankings,
           flag_price_moves
  WRITE → add_to_watchlist, remove_from_watchlist,
           save_research_note, save_analysis_report

Agentic loop:
  User query
       ↓ LLM decides which tools to call
  Tool execution (Gold tables + Vector Search + Delta writes)
       ↓ Results sent back to LLM
  LLM synthesizes response
       ↓ Repeats until no more tool calls
  Final response returned to user
""")
print("Phase 7 complete ✓")
